In [ ]:
# pip install open3d trimesh numpy scipy pandas

import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

import multiprocessing as mp
import time
import traceback
from pathlib import Path

import numpy as np
import open3d as o3d
import pandas as pd
import trimesh
from scipy.spatial import cKDTree

STL_DIR = Path("outputs/hunyuan_stls")
GT_DIR = Path("input/stls_gt")
OUT_CSV = Path("outputs/volume_surface_error.csv")

OBJECT_TO_GT = {
    "Bearing": "bearing.stl",
    "Connector": "connector.stl",
    "Mount": "mount.stl",
    "Plug": "plug.stl",
}

REG_N_POINTS = 30_000
REG_SCREEN_KEEP = 4
N_SURFACE_SAMPLES = 30_000
MIN_POLISH_SCALE, MAX_POLISH_SCALE = 0.83, 1.20
WORKERS = os.cpu_count()
COLUMNS = ["folder_path", "image_name", "env_iou", "vol_dft_pct", "surface_dft_um_mean", "surface_dft_um_std"]


def load_clean(path):
    m = o3d.io.read_triangle_mesh(str(path))
    m.remove_duplicated_vertices()
    m.remove_duplicated_triangles()
    m.remove_degenerate_triangles()
    m.remove_unreferenced_vertices()
    return m


def to_trimesh(m):
    return trimesh.Trimesh(vertices=np.asarray(m.vertices), faces=np.asarray(m.triangles), process=False)


def fpfh(pcd, voxel):
    down = pcd.voxel_down_sample(voxel)
    down.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=voxel * 2.0, max_nn=30))
    feat = o3d.pipelines.registration.compute_fpfh_feature(
        down, o3d.geometry.KDTreeSearchParamHybrid(radius=voxel * 5.0, max_nn=100)
    )
    return down, feat


def pca_axes(points):
    eigvals, eigvecs = np.linalg.eigh(np.cov(points.T))
    return eigvecs[:, np.argsort(eigvals)[::-1]]


def icp_rigid(source, target, threshold, init, max_iter=60):
    return o3d.pipelines.registration.registration_icp(
        source, target, threshold, init,
        o3d.pipelines.registration.TransformationEstimationPointToPoint(with_scaling=False),
        o3d.pipelines.registration.ICPConvergenceCriteria(
            relative_fitness=1e-7, relative_rmse=1e-7, max_iteration=max_iter
        ),
    )


def multiscale_icp_rigid(source, target, voxel, init):
    transform, result = init, None
    for mult in (8.0, 4.0, 2.0, 1.0, 0.5):
        result = icp_rigid(source, target, voxel * mult, transform)
        transform = result.transformation
    return result


def umeyama_scale(test_pcd, ref_pcd, transform, thresh):
    aligned = o3d.geometry.PointCloud(test_pcd)
    aligned.transform(transform)
    ev = o3d.pipelines.registration.evaluate_registration(aligned, ref_pcd, thresh, np.eye(4))
    corr = np.asarray(ev.correspondence_set)
    if len(corr) < 50:
        return 1.0
    sp = np.asarray(aligned.points)[corr[:, 0]]
    tp = np.asarray(ref_pcd.points)[corr[:, 1]]
    ss = float(np.sum((sp - sp.mean(axis=0)) ** 2))
    st = float(np.sum((tp - tp.mean(axis=0)) ** 2))
    return 1.0 if ss < 1e-9 else float(np.sqrt(st / ss))


def symmetric_fitness(test_pcd, ref_pcd, transform, thresh):
    reg = o3d.pipelines.registration
    fwd = reg.evaluate_registration(test_pcd, ref_pcd, thresh, transform).fitness
    aligned = o3d.geometry.PointCloud(test_pcd)
    aligned.transform(transform)
    bwd = reg.evaluate_registration(ref_pcd, aligned, thresh, np.eye(4)).fitness
    return min(fwd, bwd)


def pca_candidate_rotations(ref_pts, test_pts):
    Vr, Vt = pca_axes(ref_pts), pca_axes(test_pts)
    signs = [(1, 1, 1), (-1, 1, 1), (1, -1, 1), (1, 1, -1), (-1, -1, 1), (-1, 1, -1), (1, -1, -1), (-1, -1, -1)]
    out = []
    for Vt_o in (Vt, Vt[:, [0, 2, 1]]):
        for s in signs:
            R = Vr @ (Vt_o @ np.diag(s)).T
            if np.linalg.det(R) > 0:
                out.append(R)
    return out


def register_to_reference(ref_mesh, test_mesh, n_points=REG_N_POINTS, n_ransac_trials=3):
    ref_pcd = ref_mesh.sample_points_uniformly(n_points)
    test_pcd = test_mesh.sample_points_uniformly(n_points)
    ref_center, test_center = ref_pcd.get_center(), test_pcd.get_center()
    ref_pcd.translate(-ref_center)
    test_pcd.translate(-test_center)

    ref_pts, test_pts = np.asarray(ref_pcd.points), np.asarray(test_pcd.points)
    ref_scale0 = float(np.sqrt(np.mean(np.sum(ref_pts ** 2, axis=1))))
    test_scale0 = float(np.sqrt(np.mean(np.sum(test_pts ** 2, axis=1))))
    pre_scale = ref_scale0 / max(test_scale0, 1e-12)
    test_pcd.scale(pre_scale, center=(0.0, 0.0, 0.0))
    test_pts = np.asarray(test_pcd.points)
    voxel = max(ref_scale0 / 40.0, 1e-6)

    init_rotations = pca_candidate_rotations(ref_pts, test_pts)
    ref_down, ref_fpfh = fpfh(ref_pcd, voxel)
    reg = o3d.pipelines.registration
    for _ in range(n_ransac_trials):
        test_down, test_fpfh = fpfh(test_pcd, voxel)
        result = reg.registration_ransac_based_on_feature_matching(
            test_down, ref_down, test_fpfh, ref_fpfh,
            mutual_filter=True,
            max_correspondence_distance=voxel * 1.5,
            estimation_method=reg.TransformationEstimationPointToPoint(False),
            ransac_n=4,
            checkers=[
                reg.CorrespondenceCheckerBasedOnEdgeLength(0.9),
                reg.CorrespondenceCheckerBasedOnDistance(voxel * 1.5),
            ],
            criteria=reg.RANSACConvergenceCriteria(100_000, 0.999),
        )
        init_rotations.append(result.transformation[:3, :3])

    sym_thresh = voxel * 2.0
    screened = []
    for R in init_rotations:
        init = np.eye(4)
        init[:3, :3] = R
        coarse = icp_rigid(test_pcd, ref_pcd, voxel * 6.0, init, max_iter=30)
        screened.append((symmetric_fitness(test_pcd, ref_pcd, coarse.transformation, sym_thresh), coarse.transformation))
    screened.sort(key=lambda t: -t[0])

    best_T, best_score = np.eye(4), -1.0
    for _, coarse_T in screened[:REG_SCREEN_KEEP]:
        icp = multiscale_icp_rigid(test_pcd, ref_pcd, voxel, coarse_T)
        s = symmetric_fitness(test_pcd, ref_pcd, icp.transformation, sym_thresh)
        if s > best_score:
            best_score, best_T = s, icp.transformation

    k = min(max(umeyama_scale(test_pcd, ref_pcd, best_T, sym_thresh), MIN_POLISH_SCALE), MAX_POLISH_SCALE)
    polished_T = best_T.copy()
    polished_T[:3, :3] *= k
    score_polished = symmetric_fitness(test_pcd, ref_pcd, polished_T, sym_thresh)
    if score_polished >= best_score - 0.02:
        best_T, best_score = polished_T, score_polished

    T_center_test = np.eye(4)
    T_center_test[:3, 3] = -test_center
    T_prescale = np.eye(4)
    T_prescale[:3, :3] *= pre_scale
    T_uncenter_ref = np.eye(4)
    T_uncenter_ref[:3, 3] = ref_center
    return T_uncenter_ref @ best_T @ T_prescale @ T_center_test


def enclosed_volume(m):
    tm = to_trimesh(m)
    if not tm.is_watertight:
        tm = tm.copy()
        trimesh.repair.fill_holes(tm)
    return float(abs(tm.volume))


def heightfield(mesh, xs, ys, z_above):
    scene = o3d.t.geometry.RaycastingScene()
    scene.add_triangles(o3d.t.geometry.TriangleMesh.from_legacy(mesh))
    gx, gy = np.meshgrid(xs, ys, indexing="ij")
    origins = np.stack([gx.ravel(), gy.ravel(), np.full(gx.size, z_above)], axis=1)
    rays = np.concatenate([origins, np.tile([0.0, 0.0, -1.0], (len(origins), 1))], axis=1).astype(np.float32)
    t_hit = scene.cast_rays(o3d.core.Tensor(rays))["t_hit"].numpy()
    z_top = z_above - t_hit
    z_top[~np.isfinite(t_hit)] = np.nan
    return z_top


def volume_metrics(ref, test, xy_cell):
    rV, tV = np.asarray(ref.vertices), np.asarray(test.vertices)
    z0 = float(rV[:, 2].min())
    mins = np.minimum(rV.min(0), tV.min(0))[:2] - xy_cell
    maxs = np.maximum(rV.max(0), tV.max(0))[:2] + xy_cell
    xs = np.arange(mins[0], maxs[0] + xy_cell, xy_cell)
    ys = np.arange(mins[1], maxs[1] + xy_cell, xy_cell)
    z_above = max(rV[:, 2].max(), tV[:, 2].max()) + 5.0
    h_ref = np.nan_to_num(np.clip(heightfield(ref, xs, ys, z_above) - z0, 0, None))
    h_test = np.nan_to_num(np.clip(heightfield(test, xs, ys, z_above) - z0, 0, None))
    inter = float(np.minimum(h_ref, h_test).sum())
    union = float(np.maximum(h_ref, h_test).sum())
    v_ref, v_test = enclosed_volume(ref), enclosed_volume(test)
    return {
        "env_iou": round(inter / union, 4) if union else float("nan"),
        "vol_dft_pct": round(100.0 * abs(v_test - v_ref) / v_ref, 2) if v_ref else float("nan"),
    }


def surface_distance_um(ref, test, n_points=N_SURFACE_SAMPLES, seed=0):
    o3d.utility.random.seed(int(seed))
    ref_pts = np.asarray(ref.sample_points_uniformly(n_points).points)
    test_pts = np.asarray(test.sample_points_uniformly(n_points).points)
    d_t2r, _ = cKDTree(ref_pts).query(test_pts, k=1, workers=1)
    d_r2t, _ = cKDTree(test_pts).query(ref_pts, k=1, workers=1)
    d = np.concatenate([d_t2r, d_r2t]) * 1000.0
    return float(d.mean()), float(d.std())


_GT = {}


def get_ref(obj):
    if obj not in _GT:
        ref = load_clean(GT_DIR / OBJECT_TO_GT[obj])
        rv = np.asarray(ref.vertices)
        _GT[obj] = (ref, float(max((rv.max(0) - rv.min(0))[:2].max(), 1e-6) / 200.0))
    return _GT[obj]


def process_one(stl_path):
    stl_path = Path(stl_path)
    rel = stl_path.relative_to(STL_DIR)
    row = {"folder_path": "_".join(rel.parts[:-1]), "image_name": stl_path.stem}
    try:
        ref, xy_cell = get_ref(rel.parts[0])
        test = load_clean(stl_path)
        test.transform(register_to_reference(ref, test))
        row.update(volume_metrics(ref, test, xy_cell))
        mean_um, std_um = surface_distance_um(ref, test)
        row["surface_dft_um_mean"] = round(mean_um, 3)
        row["surface_dft_um_std"] = round(std_um, 3)
    except Exception:
        traceback.print_exc()
        row.update({k: float("nan") for k in COLUMNS[2:]})
    return row


stls = sorted(p for p in STL_DIR.rglob("*.stl") if p.relative_to(STL_DIR).parts[0] in OBJECT_TO_GT)
done = pd.read_csv(OUT_CSV) if OUT_CSV.exists() else pd.DataFrame(columns=COLUMNS)
seen = set(done["folder_path"] + "/" + done["image_name"])
todo = [str(p) for p in stls if "_".join(p.relative_to(STL_DIR).parts[:-1]) + "/" + p.stem not in seen]
print(f"{len(stls)} STLs, {len(todo)} to process with {WORKERS} workers", flush=True)

OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
rows = done.to_dict("records")
t0 = time.time()
with mp.get_context("fork").Pool(WORKERS) as pool:
    for i, row in enumerate(pool.imap_unordered(process_one, todo, chunksize=4), 1):
        rows.append(row)
        if i % 50 == 0 or i == len(todo):
            pd.DataFrame(rows)[COLUMNS].to_csv(OUT_CSV, index=False)
            print(f"{i}/{len(todo)}  {(time.time() - t0) / 60:.1f} min", flush=True)

df = pd.DataFrame(rows)[COLUMNS]
df["part"] = df["folder_path"].str.split("_").str[0]
df["texture"] = df["folder_path"].str.split("_").str[1]
print(df.groupby(["part", "texture"])[["env_iou", "vol_dft_pct", "surface_dft_um_mean"]].mean().round(2))